In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)

/Users/user/Desktop/genai/venv/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
from langchain_core.messages import HumanMessage

In [5]:
model.invoke([ HumanMessage(content="Hi,my name is Ali,and im a student at FAST university.")])

AIMessage(content='Nice to meet you, Ali. FAST university is a well-known institution in Pakistan, especially for its engineering programs. Which program are you pursuing at FAST?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 49, 'total_tokens': 81, 'completion_time': 0.063862289, 'completion_tokens_details': None, 'prompt_time': 0.002367698, 'prompt_tokens_details': None, 'queue_time': 0.005513148, 'total_time': 0.066229987}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7de6-efef-78f1-b3a7-66861310c810-0', usage_metadata={'input_tokens': 49, 'output_tokens': 32, 'total_tokens': 81})

In [6]:
from langchain_core.messages import HumanMessage,AIMessage
model.invoke(
    [
        HumanMessage(content="Hi,my name is Ali,and im a student at FAST university."),
        AIMessage(content="Nice to meet you, Ali. Congratulations on being a student at FAST University, it's a well-known and reputable institution in Pakistan. What field of study are you pursuing at FAST University?"),
        HumanMessage(content="What is my name and where do I study?")

    ]
)

AIMessage(content='Your name is Ali, and you are a student at FAST University.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 107, 'total_tokens': 122, 'completion_time': 0.015777709, 'completion_tokens_details': None, 'prompt_time': 0.006854998, 'prompt_tokens_details': None, 'queue_time': 0.005049272, 'total_time': 0.022632707}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7de6-f863-72f3-9d40-362752e705aa-0', usage_metadata={'input_tokens': 107, 'output_tokens': 15, 'total_tokens': 122})

# MESSAGE HISTORY 

we can use message history to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore.Future interactions will them load those messages and pass them into the chain as part of the input. Lets see how to use it!

In [18]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
store = {}
#to keeep chat sessions diff, we create function that will use session_id and return type will be BaseChatMessageHistory
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory() #wrt to each session id we are initializing a chatmessagehistory object, and whatever convo happens in that chat it will go in this object and store alongside session_id
    return store[session_id]





In [8]:
with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [9]:
config={"configurable":{"session_id":"chat1"}}

In [10]:
with_message_history.invoke(
    [HumanMessage(content="Hi,my name is Ali,and im a student at FAST university.")],
    config=config
)

AIMessage(content='Nice to meet you, Ali. Which branch are you studying at FAST university?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 49, 'total_tokens': 66, 'completion_time': 0.027441266, 'completion_tokens_details': None, 'prompt_time': 0.002390958, 'prompt_tokens_details': None, 'queue_time': 0.005452079, 'total_time': 0.029832224}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7de7-a564-7b63-93e0-cdabb7aae2ac-0', usage_metadata={'input_tokens': 49, 'output_tokens': 17, 'total_tokens': 66})

Internally, LangChain does this:

1️⃣ Calls:

get_session_history("chat1")


2️⃣ Gets:

store["chat1"]  → ChatMessageHistory object


3️⃣ Sends this to the model:

[ previous messages (if any) ]
+ "Hi, my name is Ali..."


4️⃣ Model responds

5️⃣ LangChain saves:

user message

AI response
inside store["chat1"]

In [11]:
with_message_history.invoke(
    [HumanMessage(content="Hi,whats my name?")],
    config=config
)

AIMessage(content="Your name is Ali, and you're a student at FAST university.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 82, 'total_tokens': 97, 'completion_time': 0.019348133, 'completion_tokens_details': None, 'prompt_time': 0.00453512, 'prompt_tokens_details': None, 'queue_time': 0.005019522, 'total_time': 0.023883253}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7de7-e5f3-7063-8b0b-cd41a63aab63-0', usage_metadata={'input_tokens': 82, 'output_tokens': 15, 'total_tokens': 97})

In [12]:
#changing the config -> changing session id
config1 = {"configurable":{"session_id":"chat2"}}
with_message_history.invoke(
    [HumanMessage(content="Hi,whats my name?")],
    config=config1
)


AIMessage(content="I'm happy to chat with you, but I don't have any information about your name. We just started this conversation, and I'm a large language model, I don't have the ability to retain information about individual users. If you'd like to share your name with me, I'd be happy to chat with you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 42, 'total_tokens': 109, 'completion_time': 0.081232442, 'completion_tokens_details': None, 'prompt_time': 0.011948009, 'prompt_tokens_details': None, 'queue_time': 0.015853264, 'total_time': 0.093180451}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7de7-f2c9-7a01-9c35-f529b3348850-0', usage_metadata={'input_tokens': 42, 'output_tokens': 67, 'total_tokens': 109})

# PROMPT TEMPLATES 

In [15]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder


Every time you call a model, you must send:

system message

previous conversation (history)

current user message

👉 MessagesPlaceholder is a slot in the prompt where chat history will be injected at runtime.

Think of it like a blank space in the prompt.

2️⃣ What exactly is MessagesPlaceholder?
MessagesPlaceholder(variable_name="messages")


It means:

“When this prompt is used, I will insert a list of chat messages here.”

That list usually contains:

Human messages

AI messages

Tool messages (if any)

3️⃣ Why we cannot hardcode messages?

❌ This is static (bad for chat apps):

("human", "Hello")
("ai", "Hi")


This never changes.

✅ We want dynamic memory:

Conversation grows

New messages added

Old messages remembered

👉 MessagesPlaceholder makes the prompt dynamic.

In [16]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer every question to your best ability"),
        MessagesPlaceholder(variable_name="messages")

    ]
)
chain = prompt|model

In [17]:
chain.invoke({"messages":[HumanMessage(content="Hi my name is ali, what about you?")]})


AIMessage(content="Nice to meet you, Ali. I don't have a personal name, but I'm an AI assistant designed to provide information and assist with tasks. I'm often referred to as a chatbot or a conversational AI. I'm here to help answer your questions, provide information, and engage in conversation to the best of my abilities. How can I assist you today, Ali?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 58, 'total_tokens': 136, 'completion_time': 0.120988342, 'completion_tokens_details': None, 'prompt_time': 0.00287733, 'prompt_tokens_details': None, 'queue_time': 0.005330247, 'total_time': 0.123865672}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7dfa-89dd-7d70-9c8a-ef34b7010d1e-0', usage_metadata={'input_tokens': 58, 'output_tokens': 78, 'total_tokens': 136})

In [19]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

In [20]:
config2 = {"configurable":{"session_id":"chat3"}}

In [21]:
with_message_history.invoke([HumanMessage(content="Hi my name is ali, what about you?")],config=config2)

AIMessage(content="Nice to meet you, Ali. I'm an AI assistant, so I don't have a personal name, but I'm here to help you with any questions or concerns you may have. I'm a machine learning model designed to provide information and assist with tasks, so feel free to ask me anything. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 58, 'total_tokens': 127, 'completion_time': 0.085343255, 'completion_tokens_details': None, 'prompt_time': 0.002880199, 'prompt_tokens_details': None, 'queue_time': 0.005388014, 'total_time': 0.088223454}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7e0b-d0cb-7c63-aded-b99a86f6cbdf-0', usage_metadata={'input_tokens': 58, 'output_tokens': 69, 'total_tokens': 127})

In [22]:
#Adding more complexity.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer every question to your best ability in {language}"),
        MessagesPlaceholder(variable_name="messages")

    ]
)
chain = prompt|model

In [24]:
response = chain.invoke({"messages":[HumanMessage(content="Hi my name is Ali")],"language":"Urdu"})
response.content

'مرہب علی صاحب، مجھے خوشی ہوئی کہ آپ نے میرا ساتھ لیا! کیا آپ کے پاس کسی خاص سوال یا مسئلے کے بارے میں پوچھنا ہے، یا آپ صرف بات چیت کرنا چاہتے ہیں؟'

In [30]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [31]:
config = {"configurable":{"session_id":"chat4"}}

In [32]:
config = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="Hi my name is Ali")],"language":"urdu"},config=config
)
response.content

'Assalamu alaikum Ali bhai, main aapki madad karne ke liye yahaan hoon. Kya aapko koi cheez maloom hone ke liye puchhna hai ya aap sirf meri madad ke liye aaye the?'

7️⃣ Why input_messages_key="messages"?

Because your input looks like this:

{
  "messages": [HumanMessage(...)]
}


LangChain needs to know:

“Which key contains chat messages?”

If you used:

input_messages_key="input"


Then input would need:

{"input":[HumanMessage(...)]}


⚠️ Key names must match exactly.

# MANAGE THE CONVERSATION HISTORY

once the histry becomes too big that it can potentially overflow the context window so itsimportsnt to limit the step size

trim_message helps to reduce how many messages we're sending to model

In [33]:
from langchain_core.messages import SystemMessage,trim_messages

In [36]:
trimmer = trim_messages(
    max_tokens = 70,
    strategy = "last", #focus on last conversation 
    token_counter = model,
    include_system=True, #include system message
    allow_partial = False, #dont allow partial info
    start_on ="human" #start from human message then go to ai response...

)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi, im ali."),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2+ 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!")
]

In [37]:
trimmer.invoke(messages)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hi, im ali.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2+ 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

we not passing config (session id) here coz its not runnablewithmessagehistory, we are managing the conversation history ourselves for this example, memory is in message list.

Memory exists only in messages variable

If Python restarts → memory gone

No sessions

No automatic storage

In [39]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)|prompt|model
)
response = chain.invoke(
    {
        "messages":messages + [HumanMessage(content="What ice cream do I like?")],
        "language":"English",


    }
)
response.content

'You like vanilla ice cream.'

now it will store info of message history and work session wise

In [40]:
#lets wrap it in the message history 
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",

)
config = {"configurable":{"session_id":"chat5"}}


In [41]:
with_message_history.invoke(
    {
        "messages": [HumanMessage(content="What is my name?")],
        "language": "English"
    },
    config={"configurable": {"session_id": "chat5"}}
)


AIMessage(content="I don't have any information about your name as our conversation has just started. You haven't shared your name with me yet. If you'd like to tell me your name, I'd be happy to learn it and address you by name in our conversation.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 55, 'total_tokens': 108, 'completion_time': 0.071857251, 'completion_tokens_details': None, 'prompt_time': 0.00269148, 'prompt_tokens_details': None, 'queue_time': 0.005175398, 'total_time': 0.074548731}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7ea2-bcef-7f13-bb64-fb5372238767-0', usage_metadata={'input_tokens': 55, 'output_tokens': 53, 'total_tokens': 108})

In [42]:
with_message_history.invoke(
    {
        "messages": [HumanMessage(content="my name is ali")],
        "language": "English"
    },
    config={"configurable": {"session_id": "chat5"}}
)


AIMessage(content="Nice to meet you, Ali. I'll make sure to address you by name from now on. Is there something specific you'd like to talk about or ask for help with? I'm here to assist you.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 121, 'total_tokens': 165, 'completion_time': 0.053900893, 'completion_tokens_details': None, 'prompt_time': 0.006759521, 'prompt_tokens_details': None, 'queue_time': 0.005202301, 'total_time': 0.060660414}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7ea3-0810-7241-848f-8d453f09fe06-0', usage_metadata={'input_tokens': 121, 'output_tokens': 44, 'total_tokens': 165})

In [43]:
with_message_history.invoke(
    {
        "messages": [HumanMessage(content="What is my name?")],
        "language": "English"
    },
    config={"configurable": {"session_id": "chat5"}}
)


AIMessage(content='Your name is Ali.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 112, 'total_tokens': 118, 'completion_time': 0.004173131, 'completion_tokens_details': None, 'prompt_time': 0.006906337, 'prompt_tokens_details': None, 'queue_time': 0.005166256, 'total_time': 0.011079468}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7ea3-2def-7570-9d60-3186e8ad9880-0', usage_metadata={'input_tokens': 112, 'output_tokens': 6, 'total_tokens': 118})

# Working with Vector store and Retriever

### DOCUMENTS

Langchain implememnts a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

1 page_content: a string representing the content;


2 metadata: a dict containing arbitrary metadata. The metadata attribute can capture information about the source of the document, its relationship to other documents,and other information. Note that an individual Document object often represents a chunk of a larger document.



In [45]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source":"mammal-pet-doc"}
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space",
        metadata={"source":"mammal-pet-doc"}
    ),
    Document(
        page_content="Goldfish are popular pets for beginners",
        metadata={"source":"fish-pet-doc"}

    ),
    Document(
        page_content="Parrots are inteligent birds capable of mimicking human speech",
        metadata={"source":"bird-pet-doc"}
    )
]

Groq needs the key passed explicitly to the client.
Hugging Face libraries auto-read the token from environment variables.

That’s the core reason.

Now let’s unpack it properly.

1️⃣ Why this works for Groq
groq_api_key = os.getenv("GROQ_API_KEY")


And later:

ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=groq_api_key
)


👉 Here YOU pass the key manually.

So:

Key lives in a Python variable

Only Groq client sees it

No need to touch os.environ

This is explicit injection.

2️⃣ Why HF is different

Hugging Face libraries are designed to:

Automatically read auth tokens from environment variables

They internally do something like:

os.getenv("HF_TOKEN")
os.getenv("HUGGINGFACEHUB_API_TOKEN")


So if you only do:

hf_token = os.getenv("HF_TOKEN")


❌ Nothing uses hf_token


❌ HF libs never see it


❌ Authentication fails

3️⃣ Why we do this
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")


This ensures:

Token is globally visible

All HF libraries can read it

No need to pass token to every call

It’s implicit authentication.

In [46]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key,model = "llama-3.1-8b-instant")


In [49]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [51]:
##VectorStore 
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents,embedding=embeddings)


In [52]:
vectorstore.similarity_search("cat")

[Document(id='6e1070f6-c8eb-49f0-a54b-7bdb384b36b2', metadata={'source': 'mammal-pet-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
 Document(id='9e943507-8eb5-4dac-8ea7-f089905c46a8', metadata={'source': 'mammal-pet-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='ddd8f089-9d42-4a61-8c80-484f11481ecc', metadata={'source': 'bird-pet-doc'}, page_content='Parrots are inteligent birds capable of mimicking human speech'),
 Document(id='a9ebeb21-5c17-4684-aedf-349e501dadac', metadata={'source': 'fish-pet-doc'}, page_content='Goldfish are popular pets for beginners')]

In [53]:
##async query
await vectorstore.asimilarity_search("cat")

[Document(id='6e1070f6-c8eb-49f0-a54b-7bdb384b36b2', metadata={'source': 'mammal-pet-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
 Document(id='9e943507-8eb5-4dac-8ea7-f089905c46a8', metadata={'source': 'mammal-pet-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='ddd8f089-9d42-4a61-8c80-484f11481ecc', metadata={'source': 'bird-pet-doc'}, page_content='Parrots are inteligent birds capable of mimicking human speech'),
 Document(id='a9ebeb21-5c17-4684-aedf-349e501dadac', metadata={'source': 'fish-pet-doc'}, page_content='Goldfish are popular pets for beginners')]

In [54]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='6e1070f6-c8eb-49f0-a54b-7bdb384b36b2', metadata={'source': 'mammal-pet-doc'}, page_content='Cats are independent pets that often enjoy their own space'),
  0.9436445832252502),
 (Document(id='9e943507-8eb5-4dac-8ea7-f089905c46a8', metadata={'source': 'mammal-pet-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706),
 (Document(id='ddd8f089-9d42-4a61-8c80-484f11481ecc', metadata={'source': 'bird-pet-doc'}, page_content='Parrots are inteligent birds capable of mimicking human speech'),
  1.5950268507003784),
 (Document(id='a9ebeb21-5c17-4684-aedf-349e501dadac', metadata={'source': 'fish-pet-doc'}, page_content='Goldfish are popular pets for beginners'),
  1.6656098365783691)]

### Retrievers 
Langchain vectorstore objects dont subclass runnable, and so cannot immediately be integrated into LCEL chains

Langchain retrienvers are runnables, so they implement a standard set of methods, and are designed to be incorporated into LCEL chains.


In [55]:
#not best way to do, the next method is best one

from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1) #.bind(k=1) means given top result (1st result)
retriever.batch(["cat","dog"])

[[Document(id='6e1070f6-c8eb-49f0-a54b-7bdb384b36b2', metadata={'source': 'mammal-pet-doc'}, page_content='Cats are independent pets that often enjoy their own space')],
 [Document(id='9e943507-8eb5-4dac-8ea7-f089905c46a8', metadata={'source': 'mammal-pet-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

as_retriever method will genenrste a Retriever, specifically a vectorstoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with follwong:

In [56]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='6e1070f6-c8eb-49f0-a54b-7bdb384b36b2', metadata={'source': 'mammal-pet-doc'}, page_content='Cats are independent pets that often enjoy their own space')],
 [Document(id='9e943507-8eb5-4dac-8ea7-f089905c46a8', metadata={'source': 'mammal-pet-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

RAG

In [57]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.
{question}
context:
{context}
"""

prompt=ChatPromptTemplate.from_messages([("human",message)])
rag_chain = {"context":retriever,"question":RunnablePassthrough()}|prompt|llm
response = rag_chain.invoke("tell me about cats")
print(response.content)

Cats are independent pets that often enjoy their own space.
